# LGTD Quick Demo

**Local Global Trend Decomposition** - A compact demonstration

This notebook demonstrates:
1. Loading synthetic datasets
2. Running LGTD decomposition
3. Visualizing results
4. Comparing with baseline methods

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from LGTD import LGTD
from LGTD.evaluation.visualization import plot_decomposition
from LGTD.evaluation.metrics import compute_mse, compute_mae

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load Synthetic Dataset

Load a pre-generated synthetic dataset with known ground truth.

In [ ]:
# Load dataset (synth1: linear trend + fixed period)
data = np.load('../data/synthetic/datasets/synth1_data.npz')

# Extract components
time = data['time']
y = data['y']
true_trend = data['trend']
true_seasonal = data['seasonal']
true_residual = data['residual']

print(f"Dataset: {data['name']}")
print(f"Trend type: {data['trend_type']}")
print(f"Period type: {data['period_type']}")
print(f"Length: {len(y)} points")

## 2. Run LGTD Decomposition

Apply LGTD to decompose the time series.

In [ ]:
# Initialize LGTD
model = LGTD(
    window_size=3,
    error_percentile=50,
    trend_selection='auto',
    verbose=True
)

# Decompose
result = model.fit_transform(y)

print(f"\nDetected periods: {result.detected_periods}")
print(f"Trend method: {result.trend_info['method']}")
print(f"Trend R²: {result.trend_info['r2']:.4f}")

## 3. Visualize Results

Compare LGTD decomposition with ground truth.

In [ ]:
# Plot decomposition with ground truth
ground_truth = {
    'trend': true_trend,
    'seasonal': true_seasonal,
    'residual': true_residual
}

plot_decomposition(
    result,
    ground_truth=ground_truth,
    title="LGTD Decomposition vs Ground Truth",
    figsize=(14, 10)
)

## 4. Evaluate Performance

Calculate decomposition errors.

In [ ]:
# Prepare results for evaluation
lgtd_result = {
    'trend': result.trend,
    'seasonal': result.seasonal,
    'residual': result.residual
}

# Compute metrics
mse = compute_mse(ground_truth, lgtd_result)
mae = compute_mae(ground_truth, lgtd_result)

# Display results
print("LGTD Performance Metrics")
print("=" * 50)
print(f"\nMean Squared Error (MSE):")
print(f"  Trend:    {mse['trend']:.6f}")
print(f"  Seasonal: {mse['seasonal']:.6f}")
print(f"  Residual: {mse['residual']:.6f}")

print(f"\nMean Absolute Error (MAE):")
print(f"  Trend:    {mae['trend']:.6f}")
print(f"  Seasonal: {mae['seasonal']:.6f}")
print(f"  Residual: {mae['residual']:.6f}")

## 5. Compare with STL Baseline

Quick comparison with the STL method.

In [ ]:
from experiments.baselines.stl import STLDecomposer

# Run STL
stl_model = STLDecomposer(period=120)
stl_result = stl_model.fit_transform(y)

# Compute STL metrics
stl_mse = compute_mse(ground_truth, stl_result)
stl_mae = compute_mae(ground_truth, stl_result)

# Comparison table
import pandas as pd

comparison = pd.DataFrame({
    'Method': ['LGTD', 'STL', 'LGTD', 'STL'],
    'Metric': ['MSE', 'MSE', 'MAE', 'MAE'],
    'Trend': [mse['trend'], stl_mse['trend'], mae['trend'], stl_mae['trend']],
    'Seasonal': [mse['seasonal'], stl_mse['seasonal'], mae['seasonal'], stl_mae['seasonal']],
    'Residual': [mse['residual'], stl_mse['residual'], mae['residual'], stl_mae['residual']]
})

print("\nLGTD vs STL Comparison")
print("=" * 70)
print(comparison.to_string(index=False))

## 6. Try Different Datasets

Test LGTD on different trend and seasonality patterns.

In [ ]:
# Load different datasets
datasets = [
    ('synth1', 'Linear + Fixed'),
    ('synth2', 'Inverted-V + Fixed'),
    ('synth4', 'Linear + Transitive'),
    ('synth7', 'Linear + Variable')
]

results_summary = []

for dataset_name, description in datasets:
    # Load dataset
    data = np.load(f'../data/synthetic/datasets/{dataset_name}_data.npz')
    y = data['y']
    
    # Run LGTD
    model = LGTD(window_size=3, error_percentile=50, verbose=False)
    result = model.fit_transform(y)
    
    # Evaluate
    gt = {'trend': data['trend'], 'seasonal': data['seasonal'], 'residual': data['residual']}
    lgtd_res = {'trend': result.trend, 'seasonal': result.seasonal, 'residual': result.residual}
    mse = compute_mse(gt, lgtd_res)
    
    results_summary.append({
        'Dataset': description,
        'Trend MSE': mse['trend'],
        'Seasonal MSE': mse['seasonal'],
        'Residual MSE': mse['residual']
    })

# Display summary
summary_df = pd.DataFrame(results_summary)
print("\nLGTD Performance Across Datasets")
print("=" * 70)
print(summary_df.to_string(index=False))

## Summary

This demo showed:
- ✓ Loading synthetic datasets with ground truth
- ✓ Running LGTD decomposition
- ✓ Visualizing decomposition results
- ✓ Evaluating decomposition accuracy
- ✓ Comparing with STL baseline
- ✓ Testing across different patterns

### Next Steps

1. Run full experiments: `python experiments/scripts/run_synthetic.py`
2. Compare all baselines: `python experiments/scripts/run_benchmarks.py`
3. Explore other notebooks for detailed analysis